# 한국어 챗봇 v3 (polyglot-ko-5.8b + QLoRA, 데이터 혼합)

58억 모델을 4bit(QLoRA)로 파인튜닝. 1.3B보다 크게 똑똑해진다. 학습은 수 시간 소요.

런타임 → GPU(T4) 설정. ⚠️ 무료 Colab은 장시간 끊길 수 있어 체크포인트를 Drive에 저장한다.

In [ ]:
!pip -q install transformers datasets accelerate peft bitsandbytes
!pip -q uninstall -y torchao

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 체크포인트 저장용 (끊겨도 이어서 학습)
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/polyglot58_ckpt'

In [ ]:
# 데이터 3종 혼합 → instruction/output 통일
from datasets import load_dataset, concatenate_datasets
def to_io(ds):
    def m(ex):
        instr=(ex.get('instruction') or '').strip()
        inp=(ex.get('input') or '').strip()
        return {'q': instr+('\n'+inp if inp else ''), 'a': str(ex.get('output') or '').strip()}
    return ds.map(m, remove_columns=ds.column_names)
parts=[]
for name,n in [('beomi/KoAlpaca-v1.1a',20000),('nlpai-lab/kullm-v2',20000),('kyujinpy/KOpen-platypus',10000)]:
    d=load_dataset(name, split='train')
    d=d.select(range(min(n,len(d))))
    parts.append(to_io(d))
data=concatenate_datasets(parts).shuffle(seed=42)
data=data.filter(lambda e: len(e['a'])>5 and len(e['q'])>3)
print('총 학습 샘플:', len(data))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
BASE='EleutherAI/polyglot-ko-5.8b'
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok=AutoTokenizer.from_pretrained(BASE); tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model=prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
rt=tok.decode(tok.encode('파이썬이 뭐야?'),skip_special_tokens=True); assert '파이썬' in rt.replace(' ',''); print('토크나이저 OK')

In [ ]:
PROMPT='### 질문: {q}\n\n### 답변: {a}'
MAX_LEN=512
def fmt(ex):
    t=PROMPT.format(q=ex['q'], a=ex['a'])+tok.eos_token
    return tok(t, truncation=True, max_length=MAX_LEN)
tokenized=data.map(fmt, remove_columns=data.column_names)
print(tokenized)

In [ ]:
lora=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['query_key_value','dense','dense_h_to_4h','dense_4h_to_h'])
model=get_peft_model(model, lora); model.print_trainable_parameters()

In [ ]:
import os
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
collator=DataCollatorForLanguageModeling(tok, mlm=False)
args=TrainingArguments(output_dir=OUT, per_device_train_batch_size=1,
    gradient_accumulation_steps=16, num_train_epochs=2, learning_rate=2e-4,
    fp16=True, warmup_ratio=0.03, logging_steps=20, save_steps=200,
    save_total_limit=2, report_to='none')
resume = os.path.isdir(OUT) and any(p.startswith('checkpoint') for p in os.listdir(OUT))
Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator).train(resume_from_checkpoint=resume)

In [ ]:
# 생성 테스트
def ask(q,n=120):
    p='### 질문: '+q+'\n\n### 답변:'
    ids=tok.encode(p,return_tensors='pt').to(model.device)
    o=model.generate(ids,max_new_tokens=n,do_sample=True,top_p=0.92,temperature=0.7,
        repetition_penalty=1.1,pad_token_id=tok.pad_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(o[0][ids.shape[1]:],skip_special_tokens=True).split('###')[0].strip()
for q in ['파이썬이 뭐야?','건강하게 사는 법 알려줘','좋은 개발자가 되려면?']:
    print('Q:',q); print('A:',ask(q)); print()

In [ ]:
# LoRA 어댑터 업로드 (5.8B는 통째로 합치면 무거워서 어댑터만 올림)
from huggingface_hub import login, create_repo
login(token='hf_본인_WRITE_토큰')
REPO='jjminu/polyglot-5.8b-chatbot-lora'
create_repo(REPO, exist_ok=True)
model.push_to_hub(REPO); tok.push_to_hub(REPO)
print('업로드 완료(어댑터):', REPO)